In [6]:

import argparse
from pathlib import Path
from typing import Tuple, List

import numpy as np
from PIL import Image
import torch

import matplotlib
matplotlib.use("Agg")  # ensure non-interactive backend
import matplotlib.pyplot as plt

from skimage import morphology, measure, util
from tqdm.auto import tqdm

IMG_EXTS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp", ".webp"}


def load_image(path: Path) -> np.ndarray:
    """Load image as RGB uint8 numpy array."""
    img = Image.open(path).convert("RGB")
    return np.asarray(img)


def load_mask_pt(path: Path, expected_shape: Tuple[int, int]) -> np.ndarray:
    """Load a binary mask saved as a .pt tensor and return bool numpy array with shape (H, W)."""
    data = torch.load(str(path), map_location="cpu")
    # Accept torch.Tensor or dict with 'mask' key
    if isinstance(data, dict) and "mask" in data:
        t = data["mask"]
    else:
        t = data
    if isinstance(t, torch.Tensor):
        arr = t.detach().cpu().numpy()
    else:
        arr = np.array(t)

    # squeeze possible channel dimension
    arr = np.squeeze(arr)
    # If mask is float or int, threshold > 0 to boolean
    if arr.dtype != bool:
        arr = arr > 0

    # Make sure shape matches (H, W); allow (H, W) or (1, H, W)
    if arr.ndim == 3 and arr.shape[0] == 1:
        arr = arr[0]
    if arr.shape != expected_shape:
        raise ValueError(f"Mask shape {arr.shape} does not match image shape {expected_shape} for {path.name}")
    return arr.astype(bool)


def keep_largest_component(mask: np.ndarray, connectivity: int = 2) -> np.ndarray:
    """Keep only the largest connected component in a boolean mask."""
    if mask.sum() == 0:
        return mask
    labeled = measure.label(mask, connectivity=connectivity)
    # areas per label (skip background 0)
    labels, counts = np.unique(labeled[labeled > 0], return_counts=True)
    if len(labels) == 0:
        return np.zeros_like(mask, dtype=bool)
    largest_label = labels[np.argmax(counts)]
    return (labeled == largest_label)


def opening_by_reconstruction(mask: np.ndarray, radius: int) -> np.ndarray:
    """Binary opening-by-reconstruction with a disk footprint."""
    if radius <= 0:
        return mask
    fp = morphology.disk(radius)
    seed = morphology.erosion(mask, footprint=fp)
    opened = morphology.reconstruction(seed, mask, method='dilation')
    return opened.astype(bool)


def closing_by_reconstruction(mask: np.ndarray, radius: int) -> np.ndarray:
    """Binary closing-by-reconstruction implemented via complement trick."""
    if radius <= 0:
        return mask
    fp = morphology.disk(radius)
    comp = np.logical_not(mask)
    seed = morphology.erosion(comp, footprint=fp)
    rec = morphology.reconstruction(seed, comp, method='dilation')
    closed = np.logical_not(rec)
    return closed.astype(bool)


def improve_mask_skimage(mask: np.ndarray, image_shape: Tuple[int, int],
                         min_obj_frac: float = 0.001,
                         min_hole_frac: float = 0.003,
                         r_open_frac: float = 0.004,
                         r_close_frac: float = 0.006,
                         r_recon_frac: float = 0.003,
                         keep_largest: bool = True) -> np.ndarray:
    """
    Apply a robust skimage.morphology cleaning pipeline:
      1) (optional) keep largest connected component
      2) remove_small_objects
      3) remove_small_holes
      4) binary opening (light)
      5) opening by reconstruction (shape-preserving)
      6) binary closing (light)
      7) closing by reconstruction (gap filling, edge-preserving)
    """
    h, w = image_shape
    area = h * w
    min_obj_area = max(64, int(min_obj_frac * area))
    min_hole_area = max(64, int(min_hole_frac * area))
    # radii scale with min(h, w)
    base = max(1, min(h, w))
    r_open = max(1, int(round(base * r_open_frac)))
    r_close = max(1, int(round(base * r_close_frac)))
    r_recon = max(1, int(round(base * r_recon_frac)))

    m = mask.astype(bool).copy()

    if keep_largest:
        m = keep_largest_component(m, connectivity=2)

    # remove speckles and pinholes
    m = morphology.remove_small_objects(m, min_size=min_obj_area, connectivity=2)
    m = morphology.remove_small_holes(m, area_threshold=min_hole_area, connectivity=2)

    # light smoothing
    m = morphology.binary_opening(m, footprint=morphology.disk(r_open))
    m = opening_by_reconstruction(m, radius=r_recon)

    m = morphology.binary_closing(m, footprint=morphology.disk(r_close))
    m = closing_by_reconstruction(m, radius=r_recon)

    # final small-hole fill (tiny perforations left after closing-by-reconstruction)
    m = morphology.remove_small_holes(m, area_threshold=min_hole_area // 2, connectivity=2)

    return m.astype(bool)


def overlay_and_save(image: np.ndarray, mask: np.ndarray, title: str, out_path: Path) -> None:
    """Save an overlay plot of mask on image (single figure)."""
    fig = plt.figure(figsize=(8, 6), dpi=150)
    plt.imshow(image)
    # overlay mask as semi-transparent; do NOT set explicit colors to follow neutral style
    plt.imshow(np.ma.masked_where(~mask, mask), alpha=0.4)
    plt.axis("off")
    plt.title(title)
    fig.tight_layout(pad=0)
    fig.savefig(out_path, bbox_inches="tight", pad_inches=0)
    plt.close(fig)


def process_dataset(images_dir: Path,
                    masks_dir: Path,
                    out_masks_dir: Path,
                    out_plots_dir: Path,
                    save_as_pt: bool = True,
                    **improve_kwargs) -> List[Tuple[str, Path, Path, Path]]:
    """
    Process all image/mask pairs. Returns list of tuples:
        (basename, improved_mask_pt_path or None, improved_mask_png_path or None, plot_dir_for_pair)
    """
    out_masks_dir.mkdir(parents=True, exist_ok=True)
    out_plots_dir.mkdir(parents=True, exist_ok=True)

    results = []

    # Build mapping from basename -> mask path (.pt)
    mask_map = {p.stem: p for p in masks_dir.glob("*.pt")}

    images = [p for p in images_dir.iterdir() if p.suffix.lower() in IMG_EXTS]
    images.sort()

    for img_path in tqdm(images, desc="Processing images"):
        base = img_path.stem
        mask_path = mask_map.get(base)
        if mask_path is None:
            print(f"[WARN] No .pt mask for image '{img_path.name}'. Skipping.")
            continue

        try:
            img = load_image(img_path)
            h, w = img.shape[:2]

            mask = load_mask_pt(mask_path, expected_shape=(h, w))

            improved = improve_mask_skimage(mask, (h, w), **improve_kwargs)

            # Save improved mask(s)
            pt_out = None
            png_out = None
            if save_as_pt:
                pt_out = out_masks_dir / f"{base}_improved.pt"
                torch.save(torch.from_numpy(improved.astype(np.uint8)), str(pt_out))

            # Save overlays (two separate plots, one per requirement)
            pair_plot_dir = out_plots_dir / base
            pair_plot_dir.mkdir(parents=True, exist_ok=True)

            overlay_and_save(img, mask, title=f"{base} — ORIGINAL mask overlay", out_path=pair_plot_dir / f"{base}_overlay_original.png")
            overlay_and_save(img, improved, title=f"{base} — IMPROVED mask overlay", out_path=pair_plot_dir / f"{base}_overlay_improved.png")

            results.append((base, pt_out, png_out, pair_plot_dir))

            print(f"[OK] {base}: saved to {pair_plot_dir}")

        except Exception as e:
            print(f"[ERROR] {base}: {e}")

    return results


# base = "./automated_underwater_area_estimation/data_preprocessed/IBF_sample/"
base = "../data_preprocessed/IBF_sample/"
images_dir = Path(base + "images/")
masks_dir = Path(base + "masks/")
out_base = Path(base)

out_masks_dir = out_base / "improved_masks"
out_plots_dir = out_base / "overlays"

results = process_dataset(
    images_dir,
    masks_dir,
    out_masks_dir,
    out_plots_dir,
    save_as_pt=True,
    min_obj_frac=0.001,
    min_hole_frac=0.003,
    r_open_frac=0.004,
    r_close_frac=0.006,
    r_recon_frac=0.003,
    keep_largest=False
)

if not results:
    print("No image/mask pairs processed. Please check your directories.")
else:
    print(f"Done. Processed {len(results)} pairs.")
    print(f"Improved masks -> {out_masks_dir}")
    print(f"Overlays per pair -> {out_plots_dir}")

Processing images:   0%|          | 0/53 [00:00<?, ?it/s]

[OK] GA 1_PB100636: saved to ..\data_preprocessed\IBF_sample\overlays\GA 1_PB100636
[OK] GA 1_PB100637: saved to ..\data_preprocessed\IBF_sample\overlays\GA 1_PB100637
[OK] GA 1_PB100638: saved to ..\data_preprocessed\IBF_sample\overlays\GA 1_PB100638
[OK] GA 1_PB100639: saved to ..\data_preprocessed\IBF_sample\overlays\GA 1_PB100639
[OK] GA 1_PB100640: saved to ..\data_preprocessed\IBF_sample\overlays\GA 1_PB100640
[OK] GA 1_PB100641: saved to ..\data_preprocessed\IBF_sample\overlays\GA 1_PB100641


KeyboardInterrupt: 